<a href="https://colab.research.google.com/github/shellyycao/IDS705_ML_Final_Project_Group10/blob/model/Resnet18_224.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResNet-18 Model Loader
### PneumoniaMNIST+
Downloads and loads the pretrained `resnet18_224_1.pth` from Zenodo.

## 1. Install & import

In [ ]:
!pip install medmnist --quiet

import os, zipfile
import requests
import torch
import torch.nn as nn
import torchvision.models as tv_models
from medmnist import INFO

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.3 MB/s eta 0:00:00
Device: cpu


## 2. Download weights from Zenodo

In [ ]:
WEIGHTS_DIR  = './weights'
WEIGHTS_FILE = os.path.join(WEIGHTS_DIR, 'resnet18_224_1.pth')
ZIP_URL      = 'https://zenodo.org/records/7782114/files/weights_pneumoniamnist.zip?download=1'
ZIP_PATH     = 'weights_pneumoniamnist.zip'

os.makedirs(WEIGHTS_DIR, exist_ok=True)

if not os.path.exists(WEIGHTS_FILE):
    print('Downloading weights zip from Zenodo...')
    response = requests.get(ZIP_URL, stream=True)
    response.raise_for_status()
    with open(ZIP_PATH, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        for member in zf.namelist():
            if member.endswith('.pth'):
                filename = os.path.basename(member)
                with zf.open(member) as src, open(os.path.join(WEIGHTS_DIR, filename), 'wb') as dst:
                    dst.write(src.read())
    print('Done.')
else:
    print(f'Already present: {WEIGHTS_FILE}')

# Show all available weight files
for f in sorted(os.listdir(WEIGHTS_DIR)):
    mb = os.path.getsize(os.path.join(WEIGHTS_DIR, f)) / 1e6
    print(f'  {f}  ({mb:.1f} MB)')

Extracting...
Done.
  resnet18_224_1.pth  (44.8 MB)
  resnet18_224_2.pth  (44.8 MB)
  resnet18_224_3.pth  (44.8 MB)
  resnet18_28_1.pth  (44.7 MB)
  resnet18_28_2.pth  (44.7 MB)
  resnet18_28_3.pth  (44.7 MB)
  resnet50_224_1.pth  (94.3 MB)
  resnet50_224_2.pth  (94.3 MB)
  resnet50_224_3.pth  (94.3 MB)
  resnet50_28_1.pth  (94.3 MB)
  resnet50_28_2.pth  (94.3 MB)
  resnet50_28_3.pth  (94.3 MB)


## 3. Build and load the model

In [ ]:
# PneumoniaMNIST is binary-class → n_classes = 2
info      = INFO['pneumoniamnist']
n_classes = len(info['label'])   # 2

# Standard torchvision ResNet-18 (official MedMNIST 224-model uses this, not the small-image variant)
model = tv_models.resnet18(weights=None, num_classes=n_classes)

# Load checkpoint — official save format: {'net': state_dict}
ckpt       = torch.load(WEIGHTS_FILE, map_location=device)
state_dict = ckpt['net'] if (isinstance(ckpt, dict) and 'net' in ckpt) else ckpt
model.load_state_dict(state_dict, strict=True)

model.to(device)
model.eval()

print(f'Model    : ResNet-18')
print(f'Classes  : {n_classes} → {info["label"]}')
print(f'Params   : {sum(p.numel() for p in model.parameters()):,}')
print(f'Device   : {device}')
print(f'Weights  : {WEIGHTS_FILE}')
print('Status   : ready ✓')

Model    : ResNet-18
Classes  : 2 → {'0': 'normal', '1': 'pneumonia'}
Params   : 11,177,538
Device   : cpu
Weights  : ./weights/resnet18_224_1.pth
Status   : ready ✓


## 4. Quick sanity check
Pass a random dummy tensor through the model to confirm it runs without errors.

In [ ]:
dummy = torch.randn(1, 3, 224, 224).to(device)   # (batch=1, RGB, H, W)

with torch.no_grad():
    logits = model(dummy)                          # (1, 2)
    probs  = torch.softmax(logits, dim=1)          # (1, 2)

print(f'Input shape  : {list(dummy.shape)}')
print(f'Output shape : {list(logits.shape)}')
print(f'Probabilities: Normal={probs[0,0]:.4f}, Pneumonia={probs[0,1]:.4f}')
print('Forward pass : OK ✓')

Input shape  : [1, 3, 224, 224]
Output shape : [1, 2]
Probabilities: Normal=0.0055, Pneumonia=0.9945
Forward pass : OK ✓
